## <span style="color: #eab308">Table of Contents</span>

- [What my_cartograph does and the resources it uses](#what-my-cartograph-does-and-the-resources-it-uses)
- [First Build: what it has and how to launch it](#first-build-what-it-has-and-how-to-launch-it)

<a id="what-my-cartograph-does-and-the-resources-it-uses"></a>

## <span style="color: #eab308">What my_cartograph does and the resources it uses</span>

Status: **plan, not built.** Nothing gets built until you say the plan is final and prompt me to build. Reference code is the original Cartograph (`cartograph-main.zip`, phases 1–10) and its specs (`project-assets.zip`, phases 1–13); the sibling `my_polaris` project is the reference for the AI call.

### What the app does

`my_cartograph` is a **local developer tool for one person on localhost**. You give it a TypeScript or JavaScript project as a **zip**, and it draws that project as an interactive **dependency map**: folders are boxes, imports are lines. It reads the code with a real parser; the AI never decides what is on the map.

**One rule everything rests on: every box and every line comes from really parsing the code.** The AI may explain what the parser found; it may not decide that two files are connected, and it may not walk the graph. A file that can't be parsed is skipped and counted; an import that can't be resolved is listed with a reason; a route is shown only if its method and full path are both written in the code. Absent beats approximate.

**Screens**

```
 LANDING PAGE  (/)                               MAP PAGE  (/projects/<id>)
┌───────────────────────────────────┐   ┌────────────────────────────────────────────────┐
│ cartograph        [system|light|dark]│   │ my-project     added 2026-10-19         [Re-run]│
├───────────────────────────────────┤   ├──────────┬─────────────────────────┬─────────────┤
│  Load a project                    │   │Categories│ Map │ Routes 12          │ summary or  │
│  [ Choose a .zip ]  [ Analyse ]    │   │· 214 files│                         │ selected    │
│  ▸ extract ▸ select ▸ parse ▸ store│   │Page routes│  folded boxes, open into │ file:       │
│                                    │   │Layouts …  │  panels with file rows   │ Structure | │
│  Stored projects                   │   │Components │  green in, amber out     │ Explanation │
│  my-project   214 files  today     │   │Unclassif. │                         │ blast radius│
│  other-app     90 files  3d ago    │   │  11rem    │      fills the middle    │   20rem     │
└───────────────────────────────────┘   └──────────┴─────────────────────────┴─────────────┘
```

| Capability | What it does |
|---|---|
| **Load a project** | One zip at a time. Unpacks it safely, then runs four named stages (`extract → select → parse → store`) and shows each stage live. A failure says why and in which stage. |
| **Stored projects list** | Below the load control, every project loaded before, with its name, size and when it was added. Click one to reopen it without loading again. Loading the same project twice opens the existing one. |
| **Parser** | ts-morph reads every `import`, `export … from`, `import("…")` with a literal, and `require("…")`. Each is resolved to a real file using the project's own `tsconfig` (aliases, index files, barrels). Every import ends as internal, external, excluded or unresolved, and every failure is listed with a reason. Fan-in and fan-out are counted per file. |
| **Framework adapters** | Detect Next.js, NestJS, Express, Docusaurus, React or none, and give files roles and "reached by" (so a page is never reported as unused). Routes for Next.js route handlers and NestJS controllers; Express and the rest show none rather than a guess. |
| **The map** | Folders fold into at most 24 boxes (the repository's shape decides the depth). Click a box to open it into a panel of file rows; click the header to fold it. Box height shows how many files depend on it. Layout is deterministic. |
| **Selection** | Select a file or box: it and its edges stay bright, the rest dims. Green is flowing in, amber is flowing out. Hover works between map and pane. |
| **Detail pane** | With nothing selected: a summary of the repository (framework, counts, most depended-on files, files nothing imports, how many files no convention identified). With a file selected: path, kind, role, length, what it imports, what imports it, exports, and every path clickable. |
| **Blast radius / Dependency chain** | Two buttons: what breaks if this file changes, and what this file needs. Two levels by default, instant, no network. |
| **Insights** | Four facts from the import graph in a collapsed panel: files nothing imports, files imported by unusually many, import cycles, files over 1,000 lines. Fixed sentences, no model, no scores. |
| **Category rail** | Roles for the detected framework in a fixed reading order, with counts. Click one and everything else dims. |
| **Route table** | Method, full pattern and declaring file, plus the routes it saw but could not read exactly, with reasons. |
| **Explain** | For a file or a folded folder, the AI writes a short paragraph from the file's source and its real neighbours. Every path in the answer becomes a link that moves the map. Answers are cached, so the second click is instant and is marked as cached. |
| **Label unknown files** | For a file no convention identified, the AI picks one of seven non-structural roles (service, repository, model, util, config, component, hook) or says none. It can never say page, route or controller. |
| **Theme** | System, light or dark, remembered. Dense developer look: small type, monospace paths, colour only for direction and kind, nothing moves unless clicked. |

**What it deliberately does not do:** no sign-in or accounts, no scores, grades or "issues found", no private-repository fetching or GitHub downloads, no languages other than TypeScript and JavaScript, no AI deciding connections.

**Later, once the above works (specs exist, no original code):** evals (a check that an explanation names no file that does not exist), and an "ask the repository" agent. Both are outside the first build.

### How a project flows through it

```
 choose zip ─► server unpacks to a temp folder (refuses absolute paths, "..", symlinks, oversized)
      │
      ▼  streamed to the page, stage by stage
 extract ─► select ─► parse ─► store
                                 │  data/<project>/  meta.json · source.zip · result.json · ai-cache.json
      ▼
 map page loads result.json, checks it against the parser's contract,
 and does all graph work in the browser (fold, layout, highlight, blast radius, insights)
      │
 Explain only ─► server ─► cache hit? ─► (miss) Groq ─► cache
```

### Resources it will use

**Stack**

| Layer | Resource | Used for |
|---|---|---|
| Framework | Next.js 16 (App Router), React 19, TypeScript (strict) | Pages, server routes, UI |
| Styling | Tailwind CSS v4 | The dense developer look, light and dark tokens |
| Parser | `ts-morph` | Reading imports and exports from real syntax trees |
| Map | `@xyflow/react` (React Flow) and `@dagrejs/dagre` | Drawing the graph and laying it out |
| AI | `ai` (Vercel AI SDK), `@ai-sdk/groq`, `zod` | One call path for explain-file, explain-folder and classify-file; `zod` for the role answer |
| Zip | One zip-reading package, **not chosen yet** | Unpacking safely and reading single files out of the stored zip |
| Package manager | pnpm | As in the original |

All of these are **not installed**. I will name each package and what it is for, and wait, before installing anything.

**AI**
- Provider: **Groq**, model **`openai/gpt-oss-20b`**, for all three tasks (as in `my_polaris`).
- Key: **`GROQ_API_KEY`** in `.env.local`. I will tell you exactly what and where when we reach the first AI call. I won't copy it and it is never written in the notebook.
- Used only by Explain and role labelling. Everything else works with no key and no network.
- Cache: keyed on a hash of the question, the model name and a prompt version, stored in the project's `ai-cache.json`. **LangSmith is dropped**, so there is no tracing; a cached answer is marked as cached instead.

**Storage**
- Plain files in a git-ignored `data/` folder, one folder per project: `meta.json`, `source.zip` (the zip as given), `result.json` (the parser's output, validated every time it is read), `ai-cache.json`. **No database.**
- Unpacked files are not kept. When Explain needs one file's source it is read out of `source.zip`, and its hash must match the one the parser stored.
- The project sits in OneDrive, so `data/` will sync. A zip is one file per project, which is why only the zip is kept.

**External services:** Groq (Explain only). Nothing else: no GitHub, no auth provider, no hosted database.

**Reference material**
- `cartograph-main.zip`: the original code, phases 1–10.
- `project-assets.zip`: the video kit, with `CLAUDE.md`, the project doc and the 13 phase specs.
- `my_polaris`: how the AI call is made (`generateText`, `Output.object`, Groq, `reasoningEffort: "low"`).
- The video "Nobody Can Read AI Code Anymore" (Cartograph, JavaScript Mastery).

**Removed from the original:** Clerk, Supabase (Postgres, row-level security, Realtime), `tar` / GitHub fetching, OpenAI, LangSmith, `proxy.ts`, organizations.

**Still to settle**
1. Which zip-reading package (to be named and asked for when we get there).
2. What counts as "the same project" when a zip is loaded twice: the hash of its contents (my recommendation, because `repo-main.zip` keeps its name while the code changes), or its file name.
3. `CLAUDE.md` needs rewriting for this plan before building: no Clerk, no Supabase, disk instead of database, Groq instead of OpenAI, no tracing.

<a id="first-build-what-it-has-and-how-to-launch-it"></a>

## <span style="color: #eab308">First Build: what it has and how to launch it</span>

The first build of `my_cartograph` exists and runs. It follows the plan above, with the changes listed under *What differs from the plan*. The code is in `my_cartograph/`, next to this notebook's folder.

### Is this a Next.js app? Yes

`my_cartograph` is a **Next.js 16.3.6 app** (App Router, Turbopack) with React 19 and TypeScript. `npm run dev` is just `next dev`, and `npm run build` is `next build`. It is not a Vite app: Next.js is the server, the bundler and the router, and the file system is the router.

| Next.js piece | Where | What it does here |
|---|---|---|
| **Layout** | `app/layout.tsx` | The shell around every page: `<html>`, fonts, and the theme read from a cookie so the first HTML already has the right theme. |
| **Pages** (server components) | `app/page.tsx`, `app/projects/[id]/page.tsx` | The landing page and the map page. They run on the server, so they can read `data/` directly. `[id]` is a dynamic route: the folder name becomes the project id. |
| **Route handler** | `app/api/projects/route.ts` | Receives the zip as the request body and streams the stage updates back, one JSON line per stage. |
| **Server actions** | `app/projects/[id]/actions.ts` | The Explain buttons call these straight from the browser; they run on the server, where the Groq key and the files are. |
| **Client components** (`"use client"`) | `components/load-form.tsx`, `analysis-view.tsx`, the map, the pane | Everything interactive. The graph work (folding, highlighting, blast radius) runs here, in the browser, with no requests. |

Unlike the Polaris app there is no Convex, Clerk or Inngest: one Next.js process and plain files in `data/`. The Next.js version here is newer than most documentation, so the rule in `AGENTS.md` applies: read the guide in `node_modules/next/dist/docs/` before relying on memory.

### How to launch it

**First time on a machine** (the packages are already installed on this one):

```powershell
cd C:\Users\rush_\OneDrive\APP_Cartograph\my_cartograph
npm install
```

**Every time:**

```powershell
cd C:\Users\rush_\OneDrive\APP_Cartograph\my_cartograph
npm run dev
```

Then open **http://localhost:3000**. To stop it, press `Ctrl+C` in that terminal. If something else already uses port 3000, Next.js says so and picks the next free one; use the address it prints.

**What you need:**
- Node 24 (this machine has 24.20). No database, no account, no Docker.
- `.env.local` in `my_cartograph/` with one line, `GROQ_API_KEY=...`. It was copied from `my_polaris/.env`. Without it everything works **except Explain** and role labelling, which say so in words. The file is git-ignored; the key is not in this notebook.

### Using it

1. On the landing page choose a zip and click **Analyse**. The steps tick past: **Upload** (percent) → **Unpack zip** → **Select files** → **Parse imports** → **Store project**. A failure names the step and says why.
2. You land on the map. The **stored projects** list on the landing page now includes it; click any entry to reopen it without loading the zip again.
3. Loading the same code twice (even from a differently named zip) opens the stored copy instead of making a second one.

**On the map:**

| Where | What to try |
|---|---|
| **Map** | Click a folded box to open it into a panel of file rows; click the panel header to fold it. Scroll the wheel inside a panel with more than 14 files. Click a file to select it: its edges light up, **green flows into it, amber flows out of it**, everything else dims. Click empty canvas to deselect. |
| **Right pane, nothing selected** | The repository summary: framework, counts, most depended-on files, files nothing imports, files no convention identified, and a collapsed **Insights** panel. |
| **Right pane, a file selected** | **Structure** tab: kind, role, folder, length, depends on, depended on by, exports, both full lists, and the **Blast radius** and **Dependency chain** buttons. **Explanation** tab: the **Explain** button. Every path is clickable and moves the map. |
| **Left rail** | Roles for the detected framework with counts. Click one to dim everything that is not in it. |
| **Routes tab** (above the map) | Method, full pattern and declaring file, plus routes it saw but could not read exactly. |
| **Theme** (top right) | system / light / dark, remembered. |

### Terminal tools

These run the same code with no web app:

```powershell
npm run load -- "C:\path\to\project.zip"      # load a zip into data/ the way the landing page does
npm run parse -- <folder>                       # parse any folder and print files, coverage, routes
npm run parse -- --read data\<id>\result.json   # re-check a stored result against the contract and print its summary
npm run typecheck                               # tsc --noEmit
npm run lint
npm run build
```

### What is stored, and where

One folder per project in `my_cartograph/data/` (git-ignored):

```
data/
  <project id>/            ← 16 hex characters, a hash of the project's files
    meta.json              ← name, when added, counts: what the list shows
    result.json            ← the parser's output, checked against its contract on every read
    sources.json           ← the text of every parsed file, so Explain can read it
    ai-cache.json          ← explanations and role labels, made on first Explain click
    model-roles.json       ← roles the model gave unclassified files
```

**The zip is deleted as soon as the run ends**, whatever the outcome; your original zip is never touched. While running, the code is unpacked into the system temp folder (not into `data/`) and removed afterwards. The folder is small: `my_polaris.zip` is 483 MB and its stored project is about 470 KB.

### What is in the code

About 6,600 lines of TypeScript. The parser, graph maths, map, detail pane and prompts are the original's design, carried over; the zip reader, the store, the ingest run and the landing page are new.

| Folder | What it is |
|---|---|
| `lib/parser/` | The standalone parser (ts-morph) and the six framework adapters. Imports no Next, React or file store. |
| `lib/graph/` | Pure graph functions: folding, view, highlight, blast radius, insights, labels. |
| `lib/zip.ts` | **New.** A zip reader on Node's own zlib (central directory, stored and deflated entries, zip64). Refuses encrypted entries, other compression, and entries that inflate larger than their header says. |
| `lib/ingest/` | **New.** `run.ts` is the four stages; `stages.ts` is the stage list the browser also reads. Refuses entries with absolute paths, `..` or drive letters; skips symlinks; skips `node_modules` and dot-folders (creating them empty so the parser still lists them as excluded); writes empty stand-ins for images and the like so imports of them still resolve. Limits: 500,000 entries, 1 GB of code and JSON, 4 GB upload. |
| `lib/store/projects.ts` | **New.** Everything under `data/`: list, read, write, cache, model roles, sources. |
| `lib/ai/` | `client.ts` is the one place a model is chosen (Groq, `openai/gpt-oss-20b`); `tasks.ts` has explain-file, explain-folder and classify-file behind the cache; `prompts.ts` is the original's. |
| `lib/analysis/context.ts` | What the model is handed, rebuilt from the stored result. The browser only names a file or folder. |
| `app/` | `/` landing page, `/projects/[id]` map page, `api/projects` (receives the zip, streams the stages back), and the Explain server actions. |
| `components/` | The map (`map/`), detail pane, rail, route table, explanation panel, load form, theme control. |
| `scripts/` | The terminal tools above. |

### What was checked

From the terminal, on `my_polaris.zip`:
- Types, lint and the production build pass.
- Loading takes about a second. 71,690 zip entries; **140 code files found, 136 parsed, 4 skipped** (all `.d.ts` files, each with its reason); **496 imports seen: 205 internal, 279 external, 12 excluded, 0 unresolved**; 205 edges.
- Detected **Next.js**; 21 files reached without an import (7 pages, 8 route handlers, 2 layouts, 1 proxy, 3 config files); **10 routes** (all in `src/app/api`).
- Explain against Groq: the first call took about 0.75 s, the second came from the cache in 2 ms. Role labelling of an unclassified file worked.
- All 136 stored sources hash to the value the parser recorded. Loading the same zip through the web endpoint leaves nothing but the project folder in `data/`.

**Not checked: anything that needs a browser** (how the map looks, clicking, scrolling, the pane). That is yours to run.

### What differs from the plan above

| Plan said | Built |
|---|---|
| Keep the zip in the project folder; Explain reads single files out of it | **Zip is deleted after loading**; `sources.json` holds the text of the parsed files instead. |
| A "stale" marker for runs that die mid-way | Not built: nothing is listed until a run finishes, so there is nothing stale to show. |
| Re-run button on a project | Not built: reload the zip. |
| "Repository moved on" freshness note | Not built: a zip has no commit. |
| Package manager pnpm | **npm** (pnpm isn't installed here). |
| A zip-reading package to be chosen | **None**: written on Node's built-ins. |
| Coverage banner with a percentage | Not built, as in the original's code. The summary shows "N skipped" and "N unresolved". |

### Not in this build

Evals, the "ask the repository" agent, an option to delete a stored project from the list (delete its folder in `data/` by hand), and anything for private repositories or other languages. `LangSmith` tracing was dropped as decided.